<a href="https://colab.research.google.com/github/HatolkarAV/ICU-Weaning-Prediction/blob/main/notebooks/Data_PreProcessing/06_PreProcessing_and_Resampling_latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Objective

* Give every visit a shared hour-0 reference point (anchor = imv_end)
* Line up the 9 hourly targets on a strict per-visit hourly grid, so the genuinely missing hours are exposed
* Put the 4 irregular labs (pao2, paco2, ph, lactate) onto the same hourly grid
* Fill short gaps with LOCF up to a per-class cutoff, beyond which gaps stay missing, and keep an is_observed mask
* Pivot long -> wide and save two artifacts: a filled grid (Tier 1/2) and an unfilled grid (Tier 3), plus updated metadata

### Google Drive setup and project folders

Mount Drive and point to the project folders. This notebook reads the files saved
by notebooks 04 and 05, and writes its own five files back to Drive.

In [1]:
# Project setup
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/ventilator_weaning')

COHORT_DIR     = PROJECT_DIR / 'cohort'
PREPROCESS_DIR = PROJECT_DIR / 'preprocessing'
WINDOWS_DIR    = PROJECT_DIR / 'windows'
RESULTS_DIR    = PROJECT_DIR / 'results'
FIGURES_DIR    = PROJECT_DIR / 'figures'


for d in [COHORT_DIR, PREPROCESS_DIR, WINDOWS_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Files this notebook reads
COHORT_PATH        = COHORT_DIR / 'cohort_zappala.parquet'
FEATURES_LONG_PATH = COHORT_DIR / 'cohort_features_long.parquet'
FEATURE_META_PATH  = COHORT_DIR / 'feature_metadata.json'

# Files this notebook writes
CLEAN_LONG_PATH   = PREPROCESS_DIR / 'cohort_features_long_clean.parquet'
ANCHORED_PATH     = PREPROCESS_DIR / 'cohort_features_anchored.parquet'
UNFILLED_PATH     = PREPROCESS_DIR / 'cohort_wide_unfilled.parquet'
FILLED_PATH       = PREPROCESS_DIR / 'cohort_wide_filled.parquet'
PREPROCESS_META_PATH = PREPROCESS_DIR / 'preprocessing_metadata.json'

# Stop early with a clear message if an input file is missing
for p in [COHORT_PATH, FEATURES_LONG_PATH, FEATURE_META_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Cannot find {p}. Run notebooks 04 and 05 first.")

print('Project folder:', PROJECT_DIR)
print('All three input files found, ready to start.')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/ventilator_weaning
All three input files found, ready to start.


Drive is mounted and the folders exist. The check stops the notebook now, with a
clear message, if any input file from notebook 04 or 05 is missing.

Note on metadata: this notebook reads notebook 05's `feature_metadata.json` but
writes its own `preprocessing_metadata.json`. Keeping them separate means
re-running notebook 05 cannot wipe the sections added here.

### Imports and configuration

**Initial BigQuery setup by Ayushi Kashyap - adapted for this notebook.**

In [2]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt
import json

In [3]:
PROJECT_ID         = 'capstoneweaningprediction' #@param {type:"string"}
DATASET_PROJECT_ID = 'amsterdamumcdb'            #@param {type:"string"}
DATASET_ID         = 'version1_5_0'              #@param {type:"string"}
LOCATION           = 'eu'                        #@param {type:"string"}

In [4]:
import os
from google.colab import auth
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
auth.authenticate_user()
print('Authenticated')

Authenticated


In [5]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable
DataTable.max_columns = 50
DataTable.max_rows    = 80000

In [6]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
def_config = bigquery.job.QueryJobConfig(
    default_dataset=DATASET_PROJECT_ID + '.' + DATASET_ID
)
bigquery_magics.context.default_query_job_config = def_config
client = bigquery.Client(
    project=PROJECT_ID, location=LOCATION,
    default_query_job_config=def_config
)
print('BigQuery client ready')

BigQuery client ready


### Section 1: Impossible values filtering

Load the notebook 05 output and map each concept_id back to its feature name so I can work with readable names.

In [7]:
# Load Notebook 05 outputs from Drive
df = pd.read_parquet(FEATURES_LONG_PATH)

with open(FEATURE_META_PATH) as f:
    metadata = json.load(f)

feature_concepts = metadata['feature_concepts']
concept_names = {v: k for k, v in feature_concepts.items()}

df['feature_name'] = df['measurement_concept_id'].map(concept_names)

print(f"Loaded {len(df):,} rows covering {df['visit_occurrence_id'].nunique():,} visits")

Loaded 5,938,450 rows covering 2,638 visits


#### Unit consistency check

Pull the units actually used per feature in our cohort, and flag any feature recorded in more than one unit - those need standardizing .

AI-assisted: unit checking logic developed with Anthropic Claude Opus 4.8

In [8]:
# Get the visit IDs from the extracted cohort
visit_ids = df['visit_occurrence_id'].unique().tolist()
visit_ids_str = ",".join(str(v) for v in visit_ids)
print(f"Cohort scope: {len(visit_ids)} visits")

# Get the concept IDs for the selected physiological features
all_concept_ids_str = ",".join(str(c) for c in feature_concepts.values())
print(f"Feature scope: {len(feature_concepts)} concepts")

# Check which measurement units are used for each feature
query = f"""
SELECT
    m.measurement_concept_id,
    c.concept_name AS feature_name,
    m.unit_concept_id,
    u.concept_name AS unit_name,
    m.unit_source_value,
    COUNT(*) AS n_readings,
    MIN(m.value_as_number) AS min_val,
    MAX(m.value_as_number) AS max_val,
    AVG(m.value_as_number) AS avg_val
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
    ON m.measurement_concept_id = c.concept_id
LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` u
    ON m.unit_concept_id = u.concept_id
WHERE m.visit_occurrence_id IN ({visit_ids_str})
  AND m.measurement_concept_id IN ({all_concept_ids_str})
  AND m.provider_id IS NOT NULL
  AND m.value_as_number IS NOT NULL
GROUP BY
    m.measurement_concept_id,
    c.concept_name,
    m.unit_concept_id,
    u.concept_name,
    m.unit_source_value
ORDER BY feature_name, n_readings DESC
"""

unit_check_all = client.query(query).to_dataframe()

# Count the number of distinct units used for each feature
n_units_per_feature = (
    unit_check_all.groupby('feature_name')['unit_concept_id']
    .nunique()
)

print("\nFeatures recorded in multiple units:")
print(n_units_per_feature[n_units_per_feature > 1])

# Inspect the extracted unit information
unit_check_all

print("\n-> only paco2, pao2 and peep show multiple units, so those are the 3 we standardize")


Cohort scope: 2638 visits
Feature scope: 13 concepts

Features recorded in multiple units:
feature_name
Carbon dioxide [Partial pressure] in Blood    2
Oxygen [Partial pressure] in Blood            2
PEEP Respiratory system --on ventilator       2
Name: unit_concept_id, dtype: int64

-> only paco2, pao2 and peep show multiple units, so those are the 3 we standardize


Only 3 concepts come in mixed units and need converting: paco2, pao2 and peep.

#### Unit standardization
Standardization approach discussed with Ayushi Kashyap.

In [9]:
target_concepts = {
    'paco2': 3013290,
    'pao2':  3027315,
    'peep':  21490855,
}

target_ids_str = ", ".join(str(c) for c in target_concepts.values())

# Retrieve the measurements and their recorded units
query = f"""
SELECT
    m.visit_occurrence_id,
    m.measurement_concept_id,
    m.measurement_datetime,
    m.value_as_number,
    m.unit_concept_id,
    u.concept_name AS unit_name
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` u
    ON m.unit_concept_id = u.concept_id
WHERE m.visit_occurrence_id IN ({visit_ids_str})
  AND m.measurement_concept_id IN ({target_ids_str})
  AND m.provider_id IS NOT NULL
  AND m.value_as_number IS NOT NULL
"""

units_df = client.query(query).to_dataframe()

# Summarize the units used for each measurement concept
unit_summary = (
    units_df
    .groupby(['measurement_concept_id', 'unit_concept_id', 'unit_name'])
    .size()
)

print(unit_summary)

units_df

measurement_concept_id  unit_concept_id  unit_name                
3013290                 8876             millimeter mercury column    215518
                        44777602         kilopascal                     5485
3027315                 8876             millimeter mercury column    212780
                        44777602         kilopascal                     5445
21490855                720858           millibar                          1
                        44777590         centimeter watercolumn       462403
dtype: int64


,visit_occurrence_id,measurement_concept_id,measurement_datetime,value_as_number,unit_concept_id,unit_name
0,18809,21490855,2013-01-07 06:31:00+00:00,12.00000000000000000000000000000000000000,720858,millibar
1,12188,3027315,2013-01-23 01:58:00+00:00,0E-38,8876,millimeter mercury column
2,14566,3027315,2013-07-31 17:32:00+00:00,18.00000000000000000000000000000000000000,8876,millimeter mercury column
3,17751,3027315,2012-12-28 18:32:00+00:00,19.00000000000000000000000000000000000000,8876,millimeter mercury column
4,13164,3027315,2013-01-15 05:36:00+00:00,22.00000000000000000000000000000000000000,8876,millimeter mercury column
...,...,...,...,...,...,...
901627,12799,21490855,2013-01-01 09:09:00+00:00,7.00000000000000000000000000000000000000,44777590,centimeter watercolumn
901628,16639,21490855,2013-01-01 16:02:00+00:00,7.00000000000000000000000000000000000000,44777590,centimeter watercolumn
901629,16639,21490855,2013-01-02 04:02:00+00:00,7.00000000000000000000000000000000000000,44777590,centimeter watercolumn
901630,16639,21490855,2013-01-01 18:02:00+00:00,7.00000000000000000000000000000000000000,44777590,centimeter watercolumn


Convert everything to one unit per feature: paco2/pao2 -> kPa, peep -> cmH2O. Rows already in the target unit pass through unchanged.

AI-assisted: unit standardization logic developed with Anthropic Claude Opus 4.8

In [10]:
# (Used AI to help structure the per-unit conversion branches)
CONCEPT_NAMES_INV = {v: k for k, v in target_concepts.items()}
units_df['feature_name'] = units_df['measurement_concept_id'].map(CONCEPT_NAMES_INV)

def standardize_value(row):
    if row['feature_name'] in ('paco2', 'pao2'):
        if row['unit_concept_id'] == 8876:          # mmHg -> kPa
            return float(row['value_as_number']) / 7.50062
        elif row['unit_concept_id'] == 44777602:    # already kPa
            return float(row['value_as_number'])
        else:
            return None

    if row['feature_name'] == 'peep':
        if row['unit_concept_id'] == 720858:        # mbar -> cmH2O
            return float(row['value_as_number']) * 1.019716
        elif row['unit_concept_id'] == 44777590:    # already cmH2O
            return float(row['value_as_number'])
        else:
            return None

    return float(row['value_as_number'])

units_df['value_standardized'] = units_df.apply(standardize_value, axis=1).astype(float)

n_unrecognized = units_df['value_standardized'].isna().sum()
print(f"Unrecognized unit rows: {n_unrecognized}")
if n_unrecognized > 0:
    print(units_df[units_df['value_standardized'].isna()]['unit_name'].value_counts())

Unrecognized unit rows: 0


Some timestamps have the same value logged twice in both units (an ETL artifact). Check they agree after conversion, then keep one row per key.

In [11]:
# Check whether duplicate measurements still agree after unit conversion
key_cols = ['visit_occurrence_id', 'measurement_concept_id', 'measurement_datetime']

agreement_check = (
    units_df
    .groupby(key_cols)['value_standardized']
    .agg(['min', 'max', 'count'])
)

agreement_check['spread'] = agreement_check['max'] - agreement_check['min']

duplicated_only = agreement_check[agreement_check['count'] > 1]

print(f"Duplicate measurement groups: {len(duplicated_only)}")
print(f"Agreement within 0.5 units: {(duplicated_only['spread'] <= 0.5).sum()}")
print(f"Agreement > 0.5 units: {(duplicated_only['spread'] > 0.5).sum()}")

# Keep one record per visit, concept and timestamp
units_df_clean = units_df.drop_duplicates(subset=key_cols, keep='first').copy()

print(f"\nRows before deduplication: {len(units_df)}")
print(f"Rows after deduplication: {len(units_df_clean)}")

Duplicate measurement groups: 10609
Agreement within 0.5 units: 10607
Agreement > 0.5 units: 2

Rows before deduplication: 901632
Rows after deduplication: 891023


In [12]:
# look at the 2 pairs that disagree by more than 0.5 after conversion
bad_keys = duplicated_only[duplicated_only['spread'] > 0.5].index

for key in bad_keys:
    visit, concept, time = key
    rows = units_df[
        (units_df['visit_occurrence_id'] == visit) &
        (units_df['measurement_concept_id'] == concept) &
        (units_df['measurement_datetime'] == time)
    ]
    print(rows[['measurement_concept_id', 'value_as_number', 'unit_name', 'value_standardized']])
    print()

        measurement_concept_id                            value_as_number  \
351308                 3013290  55.00000000000000000000000000000000000000   
637251                 3013290   5.69999980000000000000000000000000000000   

                        unit_name  value_standardized  
351308  millimeter mercury column            7.332727  
637251                 kilopascal            5.700000  

        measurement_concept_id                            value_as_number  \
113735                 3027315  78.00000000000000000000000000000000000000   
367296                 3027315   7.50000000000000000000000000000000000000   

                        unit_name  value_standardized  
113735  millimeter mercury column            10.39914  
367296                 kilopascal             7.50000  



The two duplicate pairs showed different values after unit conversion, so they were reviewed separately before deduplication.

Write the standardized values back onto the main df

In [13]:
# Merge the standardized values back into the main dataframe
merge_cols = key_cols + ['value_standardized']
df = df.merge(units_df_clean[merge_cols], on=key_cols, how='left')

# Replace the original values for the features that required unit conversion
mask = df['measurement_concept_id'].isin(target_concepts.values())
df.loc[mask, 'value_as_number'] = df.loc[mask, 'value_standardized']

# Drop the temporary column and keep the measurement values numeric
df = df.drop(columns=['value_standardized'])
df['value_as_number'] = df['value_as_number'].astype(float)

# Remove any duplicate measurements after merging
df = df.drop_duplicates(subset=key_cols, keep='first')

print(f"Rows after standardization and deduplication: {len(df)}")
print()

# Check the final value distribution for the standardized features
print(df[df['measurement_concept_id'].isin(target_concepts.values())].groupby('measurement_concept_id')['value_as_number'].describe())



Rows after standardization and deduplication: 5925778

                           count       mean       std  min       25%  \
measurement_concept_id                                                 
3013290                 215680.0   5.852672  1.449225  0.0  4.932926   
3027315                 212940.0  12.804363  5.481474  0.0  9.865851   
21490855                462403.0   8.718467  3.456428  0.0  6.000000   

                              50%        75%        max  
measurement_concept_id                                   
3013290                  5.599537   6.532793  48.395999  
3027315                 11.732363  14.398810  87.059470  
21490855                 8.000000  10.000000  44.000000  


#### Plausible range definition


In [14]:
PLAUSIBLE_RANGES = {
    'heart_rate':        (0, 300),
    'respiratory_rate':  (0, 100),
    'spo2':              (50, 100),
    'systolic_bp':       (20, 350),
    'mean_bp':           (20, 200),
    'diastolic_bp':      (10, 200),
    'fio2':              (21, 100),
    'peep':              (0, 25),
    'tidal_volume':      (50, 2000),
    'pao2':              (2, 80),
    'paco2':             (1, 15),
    'ph':                (6.5, 8.0),
    'lactate':           (0.1, 25),
}

missing_ranges = [f for f in feature_concepts if f not in PLAUSIBLE_RANGES]
print("Features missing a range:", missing_ranges if missing_ranges else "none - all 13 covered")

Features missing a range: none - all 13 covered



#### Count how many readings fall outside range

AI-assisted: plausible-range checking logic developed with Anthropic Claude Opus 4.8

In [15]:
# Check each feature against its predefined plausible range
results = []

for feature, (low, high) in PLAUSIBLE_RANGES.items():
    subset = df[df['feature_name'] == feature]

    n_total = len(subset)
    n_below = (subset['value_as_number'] < low).sum()
    n_above = (subset['value_as_number'] > high).sum()
    n_impossible = n_below + n_above

    results.append({
        'feature': feature,
        'n_total': n_total,
        'n_below_range': n_below,
        'n_above_range': n_above,
        'n_impossible': n_impossible,
        'pct_impossible': round(100 * n_impossible / n_total, 4) if n_total else 0,
    })

impossible_summary = (
    pd.DataFrame(results)
      .sort_values('pct_impossible', ascending=False)
)

print(impossible_summary.to_string(index=False))

print("\n-> tidal_volume has the most out-of-range values, everything else is tiny")


         feature  n_total  n_below_range  n_above_range  n_impossible  pct_impossible
    tidal_volume   462329           4043           1474          5517          1.1933
           paco2   215680              6            195           201          0.0932
            peep   462403              0            274           274          0.0593
            fio2   458111            220              5           225          0.0491
    diastolic_bp   665375              2            283           285          0.0428
            spo2   661472            216             12           228          0.0345
         mean_bp   665298            179              0           179          0.0269
         lactate    85589              2             10            12          0.0140
     systolic_bp   665468             81              4            85          0.0128
            pao2   212940             16              3            19          0.0089
              ph   214975              8              

#### Null out the out-of-range values


In [16]:
# Flag values outside the predefined plausible ranges
df['was_impossible'] = False
df['value_as_number_raw'] = df['value_as_number']  # retain the original measurement

for feature, (low, high) in PLAUSIBLE_RANGES.items():
    mask = (
        (df['feature_name'] == feature) &
        (
            (df['value_as_number'] < low) |
            (df['value_as_number'] > high)
        )
    )

    df.loc[mask, 'was_impossible'] = True
    df.loc[mask, 'value_as_number'] = None

n_flagged = df['was_impossible'].sum()

print(f"Values flagged as impossible: {n_flagged} ({100 * n_flagged / len(df):.4f}% of all rows)")
print()

print(df[df['was_impossible']].groupby('feature_name').size())



Values flagged as impossible: 7057 (0.1191% of all rows)

feature_name
diastolic_bp         285
fio2                 225
heart_rate             5
lactate               12
mean_bp              179
paco2                201
pao2                  19
peep                 274
ph                    11
respiratory_rate      16
spo2                 228
systolic_bp           85
tidal_volume        5517
dtype: int64


#### Save the cleaned long table and log the Section 1 decisions to metadata.

In [17]:
# Save the cleaned long table to Drive
df.to_parquet(CLEAN_LONG_PATH, index=False)

metadata['section1_summary'] = {
    'unit_standardization': {
        'paco2_pao2': 'mmHg converted to kPa (/7.50062); standardized to kPa',
        'peep': 'millibar converted to cmH2O (x1.019716); standardized to cmH2O',
        'fio2': 'confirmed stored as percent (21-100 scale), not fraction',
    },
    'plausible_ranges': PLAUSIBLE_RANGES,
    'impossible_values_omitted': int(n_flagged),
    'impossible_value_fill_strategy': 'deferred to gap-handling (LOCF, same as genuine missing hours)',
}

# This notebook's own metadata file, so notebook 05's copy stays untouched
with open(PREPROCESS_META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved {len(df):,} rows to {CLEAN_LONG_PATH}")
print(f"Metadata written to {PREPROCESS_META_PATH}")

Saved 5,925,778 rows to /content/drive/MyDrive/ventilator_weaning/preprocessing/cohort_features_long_clean.parquet
Metadata written to /content/drive/MyDrive/ventilator_weaning/preprocessing/preprocessing_metadata.json


### Section 2: Anchor point decision

Anchor is imv_end (extubation). But some visits have imv_end right at discharge, meaning no real extubation happened - check how many, and whether they're mostly deaths.

In [18]:
# Load the notebook 04 cohort from Drive to check discharge timing
cohort_check = pd.read_parquet(COHORT_PATH)

cohort_check['admission_start'] = pd.to_datetime(cohort_check['admission_start'], utc=True)
cohort_check['admission_end']   = pd.to_datetime(cohort_check['admission_end'], utc=True)
cohort_check['imv_end']         = pd.to_datetime(cohort_check['imv_end'], utc=True)

cohort_check['hours_imv_end_to_discharge'] = (
    (cohort_check['admission_end'] - cohort_check['imv_end']).dt.total_seconds() / 3600
)

print(cohort_check['hours_imv_end_to_discharge'].describe())
print()

# Show both thresholds, to document why 8h was chosen
for thr in [6, 8]:
    n = (cohort_check['hours_imv_end_to_discharge'] <= thr).sum()
    print(f"Visits where imv_end is within {thr}h of discharge: {n} ({100*n/len(cohort_check):.1f}%)")

count    2638.000000
mean      108.784888
std       218.105921
min         0.000000
25%         8.687500
50%        41.675000
75%       103.479167
max      4712.100000
Name: hours_imv_end_to_discharge, dtype: float64

Visits where imv_end is within 6h of discharge: 509 (19.3%)
Visits where imv_end is within 8h of discharge: 641 (24.3%)


Review discharge outcomes for the cohort

In [19]:
# Retrieve discharge outcomes and recorded death dates
query = f"""
SELECT
    vo.visit_occurrence_id,
    vo.discharged_to_concept_id,
    c.concept_name AS discharge_disposition,
    d.death_date
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence` vo
LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
    ON vo.discharged_to_concept_id = c.concept_id
LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.death` d
    ON vo.person_id = d.person_id
WHERE vo.visit_occurrence_id IN ({visit_ids_str})
"""

disposition_check = client.query(query).to_dataframe()

# Summarize discharge destinations and deaths
print(disposition_check['discharge_disposition'].value_counts(dropna=False))
print()
print(f"Visits with a recorded death_date: {disposition_check['death_date'].notna().sum()}")

discharge_disposition
Inpatient Visit                     1185
None                                 738
Inpatient Cardiac Care Facility      319
Inpatient Hospital                   141
Inpatient Critical Care Facility     114
No matching concept                   58
Inpatient Psychiatric Facility        42
Intensive Care                        41
Name: count, dtype: int64

Visits with a recorded death_date: 1257


Most visits had an inpatient discharge destination, while 1,257 visits had a recorded death date

#### Create discharge and mortality flags

In [20]:
# Add death date to the cohort
cohort_check = cohort_check.merge(
    disposition_check[['visit_occurrence_id', 'death_date']],
    on='visit_occurrence_id',
    how='left'
)

cohort_check['death_date'] = pd.to_datetime(
    cohort_check['death_date'],
    utc=True
)

# Check if IMV ended within 8 hours of discharge
# 8h matches the +8h forecast window, so every kept visit has a complete window
cohort_check['imv_end_near_discharge'] = (
    cohort_check['hours_imv_end_to_discharge'] <= 8
)

# Check if death occurred during the admission
cohort_check['died_this_admission'] = (
    (cohort_check['death_date'] >= cohort_check['admission_start']) &
    (cohort_check['death_date'] <= cohort_check['admission_end'] + pd.Timedelta(days=1))
)

print("IMV end near discharge:", cohort_check['imv_end_near_discharge'].sum())
print("Died during admission:", cohort_check['died_this_admission'].sum())

IMV end near discharge: 641
Died during admission: 761


#### Check deaths during admission

In [21]:
# Create a  mortality summary table
mortality_table = pd.crosstab(
    cohort_check['imv_end_near_discharge'],
    cohort_check['died_this_admission'],
    margins=True
)

mortality_table.index = ['No', 'Yes', 'Total']
mortality_table.columns = ['Survived', 'Died', 'Total']

print(f"Died during this admission: {cohort_check['died_this_admission'].sum()}")
print()
print("Mortality by IMV end near discharge")
print(mortality_table.to_string())

Died during this admission: 761

Mortality by IMV end near discharge
       Survived  Died  Total
No         1758   239   1997
Yes         119   522    641
Total      1877   761   2638


Most patients whose IMV ended within 8 hours of discharge were patients who died during the admission.

#### Apply the anchor
Drop the near-discharge visits

In [22]:
cohort_final_06 = cohort_check[~cohort_check['imv_end_near_discharge']].copy()
print("Cohort after exclusion:", len(cohort_final_06))

Cohort after exclusion: 1997


Reload the clean feature table and restrict it to the valid visits that remain after the 8-hour exclusion.

In [23]:
# Reload the cleaned table from Drive
df = pd.read_parquet(CLEAN_LONG_PATH)

df = df.merge(
    cohort_final_06[['visit_occurrence_id', 'imv_end']],
    on='visit_occurrence_id', how='inner'
)

print(f"Rows after restricting to valid cohort: {len(df):,}")
print(f"Visits remaining: {df['visit_occurrence_id'].nunique():,}")

Rows after restricting to valid cohort: 4,849,902
Visits remaining: 1,997


Compute hours_since_imv_end - the shared relative clock every visit is lined up on.

In [24]:
df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'], utc=True)
df['imv_end'] = pd.to_datetime(df['imv_end'], utc=True)

df['hours_since_imv_end'] = (
    (df['measurement_datetime'] - df['imv_end']).dt.total_seconds() / 3600
)

print(df['hours_since_imv_end'].describe())

count    4.849902e+06
mean    -1.029716e+02
std      4.358996e+02
min     -3.115900e+03
25%     -2.332500e+02
50%     -7.600000e+01
75%      3.000000e+00
max      4.705000e+03
Name: hours_since_imv_end, dtype: float64


### Review ventilator measurements after extubation

In [25]:
# Ventilator-related features
ventilator_features = ['fio2', 'peep', 'tidal_volume']

# Measurements recorded more than 48 hours after IMV ended
late_readings = df[df['hours_since_imv_end'] > 48]

affected_visits = late_readings[
    late_readings['feature_name'].isin(ventilator_features)
]['visit_occurrence_id'].unique()

print("Ventilator-setting measurements beyond 48 hours after IMV end:")
print(late_readings[late_readings['feature_name'].isin(ventilator_features)]['feature_name'].value_counts())

print()

print(f"Visits affected: {len(affected_visits)} out of {df['visit_occurrence_id'].nunique()}")

Ventilator-setting measurements beyond 48 hours after IMV end:
feature_name
peep            29272
tidal_volume    29218
fio2            29081
Name: count, dtype: int64

Visits affected: 261 out of 1997


#### Apply the window (-48h to +8h)

Apply the analysis window around IMV end

In [26]:
# Keep measurements within the selected time window
window_start, window_end = -48, 8

df_windowed = df[
    (df['hours_since_imv_end'] >= window_start) &
    (df['hours_since_imv_end'] <= window_end)
].copy()

print(f"Rows before windowing: {len(df)} -> after: {len(df_windowed)}")
print(f"Visits before: {df['visit_occurrence_id'].nunique()} -> after: {df_windowed['visit_occurrence_id'].nunique()}")

print()

# Confirm that all remaining measurements fall within the window
print(f"Readings beyond the window end: {(df_windowed['hours_since_imv_end'] > window_end).sum()}")

print(df_windowed['hours_since_imv_end'].describe())

Rows before windowing: 4849902 -> after: 909225
Visits before: 1997 -> after: 1997

Readings beyond the window end: 0
count    909225.000000
mean        -20.688620
std          15.545574
min         -48.000000
25%         -34.000000
50%         -21.000000
75%          -7.600000
max           8.000000
Name: hours_since_imv_end, dtype: float64


After applying the −48 to +8 hour window, every remaining visit was retained. The printed row and visit counts above are the values to quote.

Save the windowed checkpoint and log Section 2 to metadata.

In [27]:
# Save the anchored table to Drive
df_windowed.to_parquet(ANCHORED_PATH, index=False)

metadata['section2_anchoring'] = {
    'anchor_point': 'imv_end',
    'excluded_visits': int(cohort_check['imv_end_near_discharge'].sum()),
    'exclusion_reason': 'imv_end within 8h of discharge - no genuine post-extubation course. Threshold set to 8h to match the forecast window; 107 of the 132 visits in the 6-8h band were in-hospital deaths.',
    'cohort_after_exclusion': int(df_windowed['visit_occurrence_id'].nunique()),
    'window_start_hours': -48,
    'window_end_hours': 8,
    'rows_after_windowing': len(df_windowed),
}

with open(PREPROCESS_META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved {len(df_windowed):,} rows to {ANCHORED_PATH}")
print(f"Visits after anchoring: {df_windowed['visit_occurrence_id'].nunique():,}")

Saved 909,225 rows to /content/drive/MyDrive/ventilator_weaning/preprocessing/cohort_features_anchored.parquet
Visits after anchoring: 1,997


### Section 3: Long -> wide pivot

Assign measurements to hourly bins

In [28]:
# Assign each measurement to an hourly time bin
df_windowed['relative_hour'] = np.floor(
    df_windowed['hours_since_imv_end']
).astype(int)

# Count how many measurements fall into each visit-feature-hour bin
bin_counts = (
    df_windowed
    .groupby(['visit_occurrence_id', 'feature_name', 'relative_hour'])
    .size()
)

print(f"Total hourly bins: {len(bin_counts)}")
print(f"Bins with multiple readings: {(bin_counts > 1).sum()}")
print(f"Collision rate: {round(100 * (bin_counts > 1).sum() / len(bin_counts), 2)}%")

print("\nSummary:")
print("A small proportion of hourly bins contain multiple measurements.")

Total hourly bins: 898505
Bins with multiple readings: 10101
Collision rate: 1.12%

Summary:
A small proportion of hourly bins contain multiple measurements.


Only 1.12% of hourly bins had multiple readings, showing that duplicate measurements within the same hour were uncommon.

#### Create the hourly wide dataset

In [29]:
# Average measurements within each hourly bin
df_binned = (
    df_windowed
    .groupby(['visit_occurrence_id', 'feature_name', 'relative_hour'])['value_as_number']
    .mean()
    .reset_index()
)

# Convert the data to wide format
df_wide = df_binned.pivot_table(
    index=['visit_occurrence_id', 'relative_hour'],
    columns='feature_name',
    values='value_as_number'
).reset_index()

df_wide.columns.name = None

print("Shape:", df_wide.shape)
df_wide.head()

Shape: (104732, 15)


,visit_occurrence_id,relative_hour,diastolic_bp,fio2,heart_rate,lactate,mean_bp,paco2,pao2,peep,ph,respiratory_rate,spo2,systolic_bp,tidal_volume
0,5,-42,NaN,NaN,NaN,0.7,NaN,9.199240,16.531967,NaN,7.23,NaN,NaN,NaN,NaN
1,5,-40,75.5,45.5,133.5,0.8,90.0,4.399636,28.264330,8.5,7.39,21.0,100.0,121.5,471.5
2,5,-39,70.0,46.0,125.0,NaN,83.0,NaN,NaN,8.0,NaN,15.0,100.0,110.0,646.0
3,5,-38,57.0,41.0,96.0,1.2,65.0,6.799438,14.932099,8.0,7.27,15.0,100.0,83.0,431.0
4,5,-37,53.0,36.0,91.0,NaN,65.0,NaN,NaN,8.0,NaN,15.0,99.0,88.0,453.0


### Section 3B: Respiratory rate concept merge

##### Evaluate respiratory rate concepts across the extubation window.

#### Compare respiratory rate measurements before and after extubation

AI-assisted: RR comparison logic developed with Anthropic Claude Opus 4.8

In [30]:
# Retrieve both respiratory rate concepts
query = f"""
SELECT
    m.measurement_concept_id,
    m.visit_occurrence_id,
    m.measurement_datetime,
    m.value_as_number
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
WHERE m.visit_occurrence_id IN ({visit_ids_str})
  AND m.measurement_concept_id IN (3024171, 3007646)
  AND m.value_as_number IS NOT NULL
  AND m.provider_id IS NOT NULL
"""

rr_check = client.query(query).to_dataframe()

# Align measurements relative to IMV end
rr_check = rr_check.merge(
    cohort_final_06[['visit_occurrence_id', 'imv_end']],
    on='visit_occurrence_id',
    how='inner'
)

rr_check['measurement_datetime'] = pd.to_datetime(rr_check['measurement_datetime'], utc=True)
rr_check['imv_end'] = pd.to_datetime(rr_check['imv_end'], utc=True)

rr_check['hrs'] = (
    rr_check['measurement_datetime'] -
    rr_check['imv_end']
).dt.total_seconds() / 3600

# Keep the analysis window
in_win = rr_check[
    (rr_check['hrs'] >= -48) &
    (rr_check['hrs'] <= 8)
]

for cid, name in [(3007646, 'on-vent'), (3024171, 'generic/monitor')]:
    sub = in_win[in_win['measurement_concept_id'] == cid]

    print(f"{name} ({cid}): pre-extubation = {(sub['hrs'] < 0).sum()}, post-extubation = {(sub['hrs'] >= 0).sum()}")

on-vent (3007646): pre-extubation = 72652, post-extubation = 2023
generic/monitor (3024171): pre-extubation = 29722, post-extubation = 12174


The on-vent RR is mainly recorded before extubation, while the generic RR provides more measurements after extubation

In [31]:
# Extract measurements for the generic respiratory rate concept (3024171)
query = f"""
SELECT
    m.visit_occurrence_id,
    m.measurement_datetime,
    m.value_as_number
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
WHERE m.visit_occurrence_id IN ({visit_ids_str})
  AND m.measurement_concept_id = 3024171
  AND m.value_as_number IS NOT NULL
  AND m.provider_id IS NOT NULL
"""

rr_monitor_raw = client.query(query).to_dataframe()

# Quick check of the extracted data
print(f"rr_monitor_raw (3024171) rows: {len(rr_monitor_raw)}")
print(f"Visits covered: {rr_monitor_raw['visit_occurrence_id'].nunique()}")

rr_monitor_raw (3024171) rows: 285552
Visits covered: 2568


Apply the same value range (0–70) .

In [32]:
was_impossible = (
    (rr_monitor_raw['value_as_number'] <= 0) |
    (rr_monitor_raw['value_as_number'] > 70)
)
print("impossible values in 3024171:", was_impossible.sum(), "of", len(rr_monitor_raw))

rr_monitor_clean = rr_monitor_raw[~was_impossible].copy()

impossible values in 3024171: 6722 of 285552


### Apply the analysis window

In [33]:
# Add IMV end time
rr_monitor_clean = rr_monitor_clean.merge(
    cohort_final_06[['visit_occurrence_id', 'imv_end']],
    on='visit_occurrence_id',
    how='inner'
)

# Convert time columns to datetime
rr_monitor_clean['measurement_datetime'] = pd.to_datetime(
    rr_monitor_clean['measurement_datetime'],
    utc=True
)

rr_monitor_clean['imv_end'] = pd.to_datetime(
    rr_monitor_clean['imv_end'],
    utc=True
)

# Calculate hours from IMV end
rr_monitor_clean['hours_since_imv_end'] = (
    (rr_monitor_clean['measurement_datetime'] - rr_monitor_clean['imv_end'])
    .dt.total_seconds() / 3600
)

# Keep data from -48 to +8 hours
rr_monitor_windowed = rr_monitor_clean[
    (rr_monitor_clean['hours_since_imv_end'] >= -48) &
    (rr_monitor_clean['hours_since_imv_end'] <= 8)
].copy()

print(f"Rows in window: {len(rr_monitor_windowed)}")
print(f"Visits in window: {rr_monitor_windowed['visit_occurrence_id'].nunique()}")

Rows in window: 40779
Visits in window: 1905


The selected window contains 40,779 respiratory rate readings across 1,905 visits.

#### Bin respiratory rate by hour

Assign the respiratory rate measurements to hourly bins and calculate the mean when multiple readings are present.

In [34]:
# Assign measurements to hourly bins
rr_monitor_windowed['relative_hour'] = np.floor(
    rr_monitor_windowed['hours_since_imv_end']
).astype(int)

# Calculate the mean value for each visit and hour
rr_monitor_binned = (
    rr_monitor_windowed
    .groupby(['visit_occurrence_id', 'relative_hour'])['value_as_number']
    .mean()
    .reset_index()
    .rename(columns={'value_as_number': 'rr_monitor'})
)

print("Binned respiratory rate rows:", len(rr_monitor_binned))

Binned respiratory rate rows: 40254


#### Combine respiratory rate measurements

Combine the ventilator and monitor respiratory rate measurements into one respiratory rate feature.

AI-assisted: respiratory rate merging logic developed with Anthropic Claude Opus 4.8

In [35]:
# Rename the existing respiratory rate column
if 'rr_vent' not in df_wide.columns:
    df_wide = df_wide.rename(
        columns={'respiratory_rate': 'rr_vent'}
    )

# Remove the previous monitor column if present
df_wide = df_wide.drop(
    columns=['rr_monitor'],
    errors='ignore'
)

# Add the monitor respiratory rate data
df_wide = df_wide.merge(
    rr_monitor_binned,
    on=['visit_occurrence_id', 'relative_hour'],
    how='left'
)

# Combine the two respiratory rate values
df_wide['respiratory_rate'] = df_wide[
    ['rr_vent', 'rr_monitor']
].mean(axis=1, skipna=True)

# Remove the temporary monitor column
df_wide = df_wide.drop(columns=['rr_monitor'])

print(
    "Respiratory rate values:",
    df_wide['respiratory_rate'].notna().sum(),
    "of",
    len(df_wide)
)

Respiratory rate values: 94189 of 104732


#### Check respiratory rate coverage

Check whether respiratory rate is available at each forecast horizon.

In [36]:
# Check respiratory rate at each forecast hour
for h in [1, 4, 8]:
    n_obs = df_wide[
        (df_wide['relative_hour'] == h) &
        (df_wide['respiratory_rate'].notna())
    ].shape[0]

    print(f"+{h}h observed respiratory_rate: {n_obs}")

# Remove the separate ventilator RR column
df_wide = df_wide.drop(columns=['rr_vent'])

print("\nColumns:", df_wide.columns.tolist())



+1h observed respiratory_rate: 1201
+4h observed respiratory_rate: 1406
+8h observed respiratory_rate: 1348

Columns: ['visit_occurrence_id', 'relative_hour', 'diastolic_bp', 'fio2', 'heart_rate', 'lactate', 'mean_bp', 'paco2', 'pao2', 'peep', 'ph', 'spo2', 'systolic_bp', 'tidal_volume', 'respiratory_rate']


### Section 4: Missing values handling

#### Step 1: Create a complete hourly grid

Create one row for each visit and hour from -48 to +8 so that missing measurements can be identified.

AI-assisted: hourly grid creation logic developed with Anthropic Claude Opus 4.8

In [37]:
# Identify the feature columns
feature_cols = [
    c for c in df_wide.columns
    if c not in ['visit_occurrence_id', 'relative_hour']
]

all_visits = df_wide['visit_occurrence_id'].unique()
all_hours = range(-48, 9)

# Create all visit and hour combinations
full_index = pd.MultiIndex.from_product(
    [all_visits, all_hours],
    names=['visit_occurrence_id', 'relative_hour']
)

# Add missing visit-hour rows
df_grid = (
    df_wide
    .set_index(['visit_occurrence_id', 'relative_hour'])
    .reindex(full_index)
    .reset_index()
)

print("Shape before reindex:", df_wide.shape, "-> after:", df_grid.shape)
print("Expected:", len(all_visits), "visits x 57 hours =", len(all_visits) * 57)

Shape before reindex: (104732, 15) -> after: (113829, 15)
Expected: 1997 visits x 57 hours = 113829


#### Step 1.5: Save the unfilled hourly grid

Save a copy of the hourly grid before filling missing values. This version will be used for the irregular time-series models.

In [38]:
# Save the unfilled hourly grid to Drive
df_grid.to_parquet(UNFILLED_PATH, index=False)

print(f"Saved unfilled grid {df_grid.shape} to {UNFILLED_PATH}")

# Make a copy for missing value handling
df_filled = df_grid.copy()

Saved unfilled grid (113829, 15) to /content/drive/MyDrive/ventilator_weaning/preprocessing/cohort_wide_unfilled.parquet


Set up the feature groups

In [39]:
# Group features by measurement type
vitals = [f for f in metadata['targets'] if f not in ventilator_features]
vent_settings = ventilator_features
labs = metadata['context_only']

feature_cols = vitals + vent_settings + labs

print("Vitals:", vitals)
print("Ventilator settings:", vent_settings)
print("Labs:", labs)

Vitals: ['heart_rate', 'respiratory_rate', 'spo2', 'systolic_bp', 'mean_bp', 'diastolic_bp']
Ventilator settings: ['fio2', 'peep', 'tidal_volume']
Labs: ['pao2', 'paco2', 'ph', 'lactate']


#### Step 2: Verify + remove blackout visits
Identify visits where all features are missing for more than 4 consecutive hours.

**AI-assisted:** longest blackout run logic developed with Anthropic Claude Opus 4.8.

In [40]:
# Mark hours where all features are missing
df_filled['all_nan'] = df_filled[feature_cols].isna().all(axis=1)

# Find the longest gap between available measurements
def longest_blackout(group):
    group = group.sort_values('relative_hour')
    empty = group['all_nan'].values
    has_data = [i for i in range(len(empty)) if not empty[i]]

    if len(has_data) < 2:
        return 0

    first, last = has_data[0], has_data[-1]
    run = 0
    maxrun = 0

    for i in range(first, last + 1):
        if empty[i]:
            run += 1
            maxrun = max(maxrun, run)
        else:
            run = 0

    return maxrun

blackouts = df_filled.groupby('visit_occurrence_id').apply(longest_blackout)

visits_to_remove = blackouts[blackouts > 4].index.tolist()

print("Visits with all-feature blackout > 4h:", len(visits_to_remove))

# Check a few visits with long gaps
for v in visits_to_remove[:2]:
    vd = df_filled[df_filled['visit_occurrence_id'] == v]
    blackout_hours = vd[vd['all_nan']]

    print(f"Visit {v}: {len(blackout_hours)} fully empty hours, all NaN:", blackout_hours[feature_cols].isna().all().all())

Visits with all-feature blackout > 4h: 17
Visit 772: 16 fully empty hours, all NaN: True
Visit 1590: 16 fully empty hours, all NaN: True


/tmp/ipykernel_572/1031493532.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  blackouts = df_filled.groupby('visit_occurrence_id').apply(longest_blackout)


In [41]:
before = df_filled['visit_occurrence_id'].nunique()
df_filled = df_filled[~df_filled['visit_occurrence_id'].isin(visits_to_remove)].copy()
df_filled = df_filled.drop(columns=['all_nan'])
after = df_filled['visit_occurrence_id'].nunique()

print("visits before:", before, "-> after:", after)
print("rows now:", len(df_filled), "(expected", after, "x 57 =", after*57, ")")

visits before: 1997 -> after: 1980
rows now: 112860 (expected 1980 x 57 = 112860 )


#### Step 3: Snapshot the is_observed mask (before any filling)
Create a missingness mask before filling the data. A value of 1 indicates an observed measurement, while 0 indicates a missing value. The mask is retained for model input and evaluation.

In [42]:
# Create an observed-value flag for each feature
for feat in feature_cols:
    df_filled[f'{feat}_is_observed'] = df_filled[feat].notna().astype(int)

# Check the percentage of observed values
print("Observed rate per feature:")

for feat in feature_cols:
    print(f"  {feat:<18} {100 * df_filled[f'{feat}_is_observed'].mean():5.1f}%")

Observed rate per feature:
  heart_rate          89.3%
  respiratory_rate    83.0%
  spo2                84.6%
  systolic_bp         87.9%
  mean_bp             87.9%
  diastolic_bp        87.9%
  fio2                64.6%
  peep                65.3%
  tidal_volume        64.3%
  pao2                26.9%
  paco2               27.2%
  ph                  27.1%
  lactate             11.7%


#### Step 4: Stage-1 LOCF with per-class cutoffs

Forward-fill missing values within each visit, using a **4-hour limit for vitals and respiratory rate and a 12-hour limit for ventilator settings and labs**. Any remaining missing values are handled in Notebook 07.


In [43]:
# Set the forward-fill limit for each feature group
cutoffs = {}

for f in vitals:
    cutoffs[f] = 4

for f in vent_settings:
    cutoffs[f] = 12

for f in labs:
    cutoffs[f] = 12

# Sort each visit by hour
df_filled = df_filled.sort_values(
    ['visit_occurrence_id', 'relative_hour']
).reset_index(drop=True)

# Forward-fill within each visit
for feat, limit in cutoffs.items():
    df_filled[feat] = (
        df_filled.groupby('visit_occurrence_id')[feat]
        .ffill(limit=limit)
    )

# Check remaining missing values
print("Remaining NaN percentage after forward fill:")

for feat in feature_cols:
    print(f"  {feat:<18} {100 * df_filled[feat].isna().mean():5.1f}%")

Remaining NaN percentage after forward fill:
  heart_rate           4.2%
  respiratory_rate     6.2%
  spo2                 4.8%
  systolic_bp          5.3%
  mean_bp              5.3%
  diastolic_bp         5.3%
  fio2                 8.5%
  peep                 8.2%
  tidal_volume         8.5%
  pao2                 6.2%
  paco2                6.0%
  ph                   6.1%
  lactate             52.7%


### Section 5: Create the final frame
The final dataset contains visit ID, relative hour, 13 feature columns, and 13 observed-value mask columns.

In [44]:
id_cols = ['visit_occurrence_id', 'relative_hour']
observed_cols = [f'{f}_is_observed' for f in feature_cols]

# Keep the required columns
df_final = df_filled[
    id_cols + feature_cols + observed_cols
].copy()

print("Final shape:", df_final.shape)
print("Columns:", df_final.columns.tolist())

Final shape: (112860, 28)
Columns: ['visit_occurrence_id', 'relative_hour', 'heart_rate', 'respiratory_rate', 'spo2', 'systolic_bp', 'mean_bp', 'diastolic_bp', 'fio2', 'peep', 'tidal_volume', 'pao2', 'paco2', 'ph', 'lactate', 'heart_rate_is_observed', 'respiratory_rate_is_observed', 'spo2_is_observed', 'systolic_bp_is_observed', 'mean_bp_is_observed', 'diastolic_bp_is_observed', 'fio2_is_observed', 'peep_is_observed', 'tidal_volume_is_observed', 'pao2_is_observed', 'paco2_is_observed', 'ph_is_observed', 'lactate_is_observed']


### Section 6: Sanity checks

#### 6.1 Grid complete and rectangular

In [45]:
rows_per_visit = df_final.groupby('visit_occurrence_id').size()
print("all visits have exactly 57 rows?", (rows_per_visit == 57).all())
print("visits:", df_final['visit_occurrence_id'].nunique(), "| total rows:", len(df_final))

all visits have exactly 57 rows? True
visits: 1980 | total rows: 112860


#### 6.2 Percentage imputed per feature

In [46]:
# Compare observed and filled values for each feature
print(f"{'Feature':<18} {'Observed %':>12} {'Filled %':>12}")
print("-" * 44)

for feat in feature_cols:
    obs_pct = 100 * df_final[f'{feat}_is_observed'].mean()

    print(f"{feat:<18} {obs_pct:>11.1f}% {100 - obs_pct:>11.1f}%")

Feature              Observed %     Filled %
--------------------------------------------
heart_rate                89.3%        10.7%
respiratory_rate          83.0%        17.0%
spo2                      84.6%        15.4%
systolic_bp               87.9%        12.1%
mean_bp                   87.9%        12.1%
diastolic_bp              87.9%        12.1%
fio2                      64.6%        35.4%
peep                      65.3%        34.7%
tidal_volume              64.3%        35.7%
pao2                      26.9%        73.1%
paco2                     27.2%        72.8%
ph                        27.1%        72.9%
lactate                   11.7%        88.3%


#### 6.3 Check one visit's values and observed flags

In [47]:
# Select one visit for review
sample_visit = df_final['visit_occurrence_id'].iloc[0]

cols_to_show = [
    'relative_hour',
    'heart_rate',
    'heart_rate_is_observed',
    'lactate',
    'lactate_is_observed'
]

# Display the first 15 hours
print(f"Visit {sample_visit}:")

print(df_final[df_final['visit_occurrence_id'] == sample_visit][cols_to_show].head(15).to_string(index=False))

Visit 5:
 relative_hour  heart_rate  heart_rate_is_observed  lactate  lactate_is_observed
           -48         NaN                       0      NaN                    0
           -47         NaN                       0      NaN                    0
           -46         NaN                       0      NaN                    0
           -45         NaN                       0      NaN                    0
           -44         NaN                       0      NaN                    0
           -43         NaN                       0      NaN                    0
           -42         NaN                       0     0.70                    1
           -41         NaN                       0     0.70                    0
           -40       133.5                       1     0.80                    1
           -39       125.0                       1     0.80                    0
           -38        96.0                       1     1.20                    1
           -37     

#### 6.4 Check the tidal volume distribution.

In [48]:
tv_observed = df_final[df_final['tidal_volume_is_observed'] == 1]['tidal_volume']
print(tv_observed.describe())

count    72520.000000
mean       484.806035
std        177.162407
min         50.000000
25%        379.000000
50%        461.000000
75%        563.000000
max       1993.000000
Name: tidal_volume, dtype: float64


Tidal volume values are mostly around 460 mL, with the majority between 380 and 560 mL.

#### 6.5 Observed counts at each forecast horizon


In [49]:
horizons = [1, 4, 8]
print(f"{'feature':<18}", end='')
for h in horizons:
    print(f"  obs@+{h}h", end='')
print("\n" + "-" * 45)

for feat in feature_cols:
    print(f"{feat:<18}", end='')
    for h in horizons:
        n_obs = df_final[(df_final['relative_hour'] == h) & (df_final[f'{feat}_is_observed'] == 1)].shape[0]
        print(f"  {n_obs:>6}", end='')
    print()

feature             obs@+1h  obs@+4h  obs@+8h
---------------------------------------------
heart_rate            1765    1788    1667
respiratory_rate      1197    1398    1340
spo2                  1500    1655    1472
systolic_bp           1724    1746    1592
mean_bp               1724    1749    1597
diastolic_bp          1723    1747    1590
fio2                     3       0       0
peep                     4       1       0
tidal_volume             3       1       0
pao2                   823     462      16
paco2                  834     466      16
ph                     829     464      16
lactate                337     184       9


Vitals remain well observed across all forecast horizons, while ventilator settings and laboratory measurements become very sparse after IMV ends.


### Section 7: Save artifacts + metadata

Two deliverables from this notebook:
* `cohort_wide_unfilled.parquet` - gaps intact, for Tier 3 irregularity-native models (saved in Section 4)
* `cohort_wide_filled.parquet` - LOCF-filled + is_observed mask, for Tier 1/2

In [50]:
# Save the filled hourly grid to Drive
df_final.to_parquet(FILLED_PATH, index=False)

print(f"Saved filled grid {df_final.shape} to {FILLED_PATH}")

metadata['section6_preprocessing'] = {
    'rr_merge': 'averaged 3007646 (on-vent) + 3024171 (monitor) per hour; each binned independently first; monitor concept zero-artifacts dropped',
    'grid': '-48h to +8h, 57 hourly slots per visit',
    'cohort_final': int(df_final['visit_occurrence_id'].nunique()),
    'locf_cutoffs': {'vitals_rr': 4, 'vent_settings': 12, 'labs': 12},
    'mask_column': 'is_observed (1=measured, 0=imputed); used as model input and to exclude imputed cells from evaluation',
    'artifacts': [str(UNFILLED_PATH.name), str(FILLED_PATH.name)],
}

with open(PREPROCESS_META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata updated: {PREPROCESS_META_PATH}")

# Read everything back so we know the files really saved
for p in [UNFILLED_PATH, FILLED_PATH]:
    n = len(pd.read_parquet(p))
    print(f"  {p.name}: {n:,} rows confirmed on Drive")

print()
print(f"FINAL COHORT: {df_final['visit_occurrence_id'].nunique():,} visits")

Saved filled grid (112860, 28) to /content/drive/MyDrive/ventilator_weaning/preprocessing/cohort_wide_filled.parquet
Metadata updated: /content/drive/MyDrive/ventilator_weaning/preprocessing/preprocessing_metadata.json
  cohort_wide_unfilled.parquet: 113,829 rows confirmed on Drive
  cohort_wide_filled.parquet: 112,860 rows confirmed on Drive

FINAL COHORT: 1,980 visits
